# 03 - RT-DETR-R18 (depot officiel lyuwenyu/RT-DETR)

Le depot est clone, ses dependances installees, et deux transforms ajoutees a son registre (flip vertical, ColorJitter).

**Variante par defaut : `rtdetr_v2`** (`rtdetrv2_pytorch`, backbone R18). La version v1 (`rtdetr_pytorch`, le RT-DETR-R18 de l'article) importe `torchvision.datapoints`, supprime depuis torchvision 0.17 : elle ne demarre pas sur un Colab recent. Pour l'essayer malgre tout : `rtdetr_runner.run_cv(variant='rtdetr_v1')` dans un environnement avec torchvision <= 0.16.

In [ ]:
# --- Installation (le depot installe ses propres dependances au 1er appel) ---
!pip install -q pycocotools wandb

In [ ]:
# --- Connexion Drive ---
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- Code du benchmark (package aphids_det) ---
import os, sys
REPO_DIR = "/content/aphids_detection"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/EmmaDub/aphids_detection.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
sys.path.insert(0, REPO_DIR)
import aphids_det
print("aphids_det", aphids_det.__version__)

In [ ]:
# --- Configuration ---
# Les chemins par defaut sont ceux de aphids_det/config.py. Pour les changer,
# decommenter et adapter, puis relancer cfg.refresh().
from pathlib import Path
import aphids_det.config as cfg

# cfg.BASE_DIR  = Path("/content/drive/MyDrive/.../tuile_viz02_640_128")
# cfg.OUT_DIR   = Path("/content/drive/MyDrive/.../puceron_model_article/data")
# cfg.EPOCHS    = 30
# cfg.USE_WANDB = True

cfg.refresh()
cfg.summary()

In [ ]:
# --- Construction des folds (symlinks locaux, a refaire a chaque session Colab) ---
from aphids_det import folds
folds.build_folds()

In [ ]:
# --- Suivi W&B (facultatif : mettre cfg.USE_WANDB = False pour s'en passer) ---
if cfg.USE_WANDB:
    import wandb
    wandb.login()
    os.environ["WANDB_PROJECT"] = cfg.WANDB_PROJECT
    print("W&B -> projet", cfg.WANDB_PROJECT)

In [ ]:
from aphids_det.runners import rtdetr_runner
df = rtdetr_runner.run_cv()           # variante rtdetr_v2 (R18)

In [ ]:
# --- Etat du CSV de benchmark ---
import pandas as pd
d = pd.read_csv(cfg.CSV_CV)
print(d.groupby("modele")["fold"].count().to_string(), "\n")
d[["modele", "fold", "map50_macro", "map5095_macro", "latency_cpu_ms",
   "n_params_M", "train_time_s"]].tail(10)